In [1]:
!pip install pandas sqlalchemy psycopg2-binary

In [2]:
# ============================================
# STEP 3: Load all CSV files into PostgreSQL (credit_risk_db)
# ============================================

import pandas as pd                          # read CSVs into DataFrames
from sqlalchemy import create_engine          # connect Python to PostgreSQL
from urllib.parse import quote_plus           # safely encode special characters in password
import glob                                   # grab all transactions_part_*.csv files at once

# --- Connection details ---
host     = "localhost"
port     = "5432"
database = "credit_risk_db"                   # this project's DB (not Vendor_Analysis)
user     = "postgres"
password = quote_plus("Postgre@Roni*1993#")   # quote_plus escapes @ # * so the URL doesn't break

# --- Build the engine (reusable connection pool) ---
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")

# Folder where CSVs live, relative to this notebook
DATA_DIR = "Data"

# --- Reusable loader function: reads one CSV, inserts into matching Postgres table ---
def load(csv_path, table_name, date_cols=None, chunksize=50000):
    print(f"Loading {csv_path} -> {table_name} ...")
    df = pd.read_csv(csv_path, parse_dates=date_cols) if date_cols else pd.read_csv(csv_path)
    df.to_sql(table_name, engine, if_exists="append", index=False, chunksize=chunksize, method="multi")
    print(f"  done: {len(df)} rows")

# --- Load parent tables first (foreign key dependency order) ---
load(f"{DATA_DIR}/customer_master.csv", "customer_master")
load(f"{DATA_DIR}/account_master.csv", "account_master", date_cols=["account_open_date"])
load(f"{DATA_DIR}/monthly_account_snapshot.csv", "monthly_account_snapshot", date_cols=["snapshot_month"])
load(f"{DATA_DIR}/fraud_alerts.csv", "fraud_alerts")
load(f"{DATA_DIR}/collections_activity.csv", "collections_activity")
load(f"{DATA_DIR}/macroeconomic_indicators.csv", "macroeconomic_indicators", date_cols=["month"])
load(f"{DATA_DIR}/default_model_dataset.csv", "default_model_dataset")

# --- Load the 4 transaction files (~1M rows total) into the same 'transactions' table ---
for f in sorted(glob.glob(f"{DATA_DIR}/transactions_part_*.csv")):
    load(f, "transactions", date_cols=["transaction_date"])

print("\nAll tables loaded successfully.")

Loading Data/customer_master.csv -> customer_master ...
  done: 20000 rows
Loading Data/account_master.csv -> account_master ...
  done: 20000 rows
Loading Data/monthly_account_snapshot.csv -> monthly_account_snapshot ...
  done: 720000 rows
Loading Data/fraud_alerts.csv -> fraud_alerts ...
  done: 50000 rows
Loading Data/collections_activity.csv -> collections_activity ...
  done: 100000 rows
Loading Data/macroeconomic_indicators.csv -> macroeconomic_indicators ...
  done: 36 rows
Loading Data/default_model_dataset.csv -> default_model_dataset ...
  done: 20000 rows
Loading Data\transactions_part_1.csv -> transactions ...
  done: 250000 rows
Loading Data\transactions_part_2.csv -> transactions ...
  done: 250000 rows
Loading Data\transactions_part_3.csv -> transactions ...
  done: 250000 rows
Loading Data\transactions_part_4.csv -> transactions ...
  done: 250000 rows

All tables loaded successfully.
